<a href="https://colab.research.google.com/github/SheffieldML/notebook/blob/main/GPy/f1_2011_qualifying_cutoff.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Skew Likelihood for F1 qualifying cut-off time prediction with a driver/team GP

In 2011 at the University of Sheffield we did some work with an F1 team predicting
cut-off times using Gaussian processes. This notebook recreates that work as a
demonstration of **skewed likelihoods** in GPy. It uses a Gaussian process whose
covariance is built from practice pace, **team**, and **driver** consistency terms,
and compares a symmetric Gaussian observation model with the O’Hagan / Azzalini
`SkewGaussian` likelihood ([GPy CIP-0014](https://github.com/SheffieldML/GPy)).


## Origin

The original code predicted qualifying cut-offs
from free-practice times:
$$
y_i = \log q_i - m\,\log p_i
$$
with inputs
$$
X_i = [\mathrm{driver},\ \mathrm{team},\ p_i,\ \log p_i,\ \mathrm{noise}_i].
$$
The compound covariance is

1. **Linear** on $\log p$ — calibration slope $m$ uncertainty
2. **RBF** on $\log p$ — smooth nonlinearity in pace
3. **Bias** — track-wide consistency
4. **IndexARD** on team — one variance per constructor
5. **IndexARD** on driver — fixed drivers pinned to their observed pace
6. **Diag** on a noise column — heteroscedastic driver consistency for the rest

Lap-time residuals are **not** symmetric: traffic and mistakes make a car more often
*slower* than a latent true pace than faster. A Gaussian observation model understates
that right-hand tail; `SkewGaussian` puts extra mass on $y > f$ when $\alpha > 0$.

This notebook implements the compound covariance in GPy (`IndexARD` / `DiagFromInput`
match GPmat `indexard` / `diag`), loads Silverstone 2011 via
`pods.datasets.f1_2011_practice_qualifying`, fits **Gaussian** and **SkewGaussian**
models side by side, and reports the predicted **P17 cut-off** distribution
(Q1 → Q2 in a 24-car field).


## Setup

Needs GPy with CIP-0014 (`GPy.likelihoods.SkewGaussian`) and pods with CIP-0005
(`f1_2011_practice_qualifying`). The next cell installs both if missing (GPy from
SheffieldML/GPy `main` until the next PyPI release ships skew; pods from the ods
CIP-0005 branch until that loader is on PyPI). Tutorial CI uses the same sources
via `installs.yml`.


In [ ]:
import importlib
import json
import shutil
import subprocess
import sys
from urllib.request import urlopen

# Shared defaults: lamd CIP-001C catalog. Temporary pins match installs.yml.
_CATALOG_JSON = (
    "https://raw.githubusercontent.com/lawrennd/lamd/main/lamd/installs/packages.json"
)
_OVERRIDES = {
    # CIP-0014 (SkewGaussian) is on GPy main; not yet in PyPI 1.15.0.
    "GPy": "git+https://github.com/SheffieldML/GPy.git#egg=gpy[plotting]",
    "pods": "git+https://github.com/lawrennd/ods.git@cip0005-f1-practice-qualifying",
}


def _pip_install(spec):
    if shutil.which("uv"):
        cmd = ["uv", "pip", "install", "--python", sys.executable, "-q", spec]
    else:
        cmd = [sys.executable, "-m", "pip", "install", "-q", spec]
    subprocess.check_call(cmd)


def _spec(name):
    if name in _OVERRIDES:
        return _OVERRIDES[name]
    try:
        with urlopen(_CATALOG_JSON, timeout=10) as response:
            packages = json.loads(response.read().decode("utf-8")).get("packages", {})
        return packages.get(name, name)
    except Exception:
        return name


def _ensure(import_name, install_as=None):
    try:
        return importlib.import_module(import_name)
    except ImportError:
        _pip_install(_spec(install_as or import_name))
        return importlib.import_module(import_name)


def _reimport(name):
    for key in list(sys.modules):
        if key == name or key.startswith(name + "."):
            del sys.modules[key]
    return importlib.import_module(name)


np = _ensure("numpy")
_ensure("matplotlib")
from matplotlib import pyplot as plt

GPy = _ensure("GPy")
pods = _ensure("pods")

# Old PyPI GPy lacks SkewGaussian — force CIP-0014 build from main.
if not hasattr(GPy.likelihoods, "SkewGaussian"):
    _pip_install(_OVERRIDES["GPy"])
    GPy = _reimport("GPy")
    if not hasattr(GPy.likelihoods, "SkewGaussian"):
        raise ImportError(
            "GPy.likelihoods.SkewGaussian still missing after "
            f"installing {_OVERRIDES['GPy']!r}"
        )

# Old PyPI pods lacks the 2011 F1 loader — force the CIP-0005 build.
if not hasattr(pods.datasets, "f1_2011_practice_qualifying"):
    _pip_install(_OVERRIDES["pods"])
    pods = _reimport("pods")
    if not hasattr(pods.datasets, "f1_2011_practice_qualifying"):
        raise ImportError(
            "pods.datasets.f1_2011_practice_qualifying still missing after "
            f"installing {_OVERRIDES['pods']!r}"
        )

pods.access.overide_manual_authorize = True

print("GPy OK; SkewGaussian available")
print("pods OK; f1_2011_practice_qualifying available")


## Local kernels

GPy has no stock IndexARD. These match the MATLAB semantics used in
`createGpModel`: same discrete label → shared scale; diagonal-from-input for
per-driver noise.


In [ ]:
class IndexARD(GPy.kern.Kern):
    """Covariance: K_ij = scale[x_i] if round(x_i)==round(x_j) else 0."""

    def __init__(self, scales_by_index, active_dims=None, name="indexard"):
        super().__init__(input_dim=1, active_dims=active_dims, name=name)
        self.scales_by_index = {int(k): float(v) for k, v in scales_by_index.items()}

    def K(self, X, X2=None):
        x = np.round(np.asarray(X)).astype(int).ravel()
        x2 = x if X2 is None else np.round(np.asarray(X2)).astype(int).ravel()
        sx = np.array([self.scales_by_index.get(int(i), 0.0) for i in x], dtype=float)
        return (x[:, None] == x2[None, :]) * sx[:, None]

    def Kdiag(self, X):
        x = np.round(np.asarray(X)).astype(int).ravel()
        return np.array([self.scales_by_index.get(int(i), 0.0) for i in x], dtype=float)

    def update_gradients_full(self, dL_dK, X, X2=None):
        pass

    def update_gradients_diag(self, dL_dKdiag, X):
        pass


class DiagFromInput(GPy.kern.Kern):
    """Covariance: K_ii = variance * exp(x_i), off-diagonal 0 (GPmat diag)."""

    def __init__(self, variance=1.0, active_dims=None, name="diag"):
        super().__init__(input_dim=1, active_dims=active_dims, name=name)
        self.variance = float(variance)

    def K(self, X, X2=None):
        if X2 is not None:
            return np.zeros((X.shape[0], X2.shape[0]))
        v = self.variance * np.exp(np.asarray(X, dtype=float).ravel())
        v = np.where(np.isfinite(v), v, 0.0)
        return np.diag(v)

    def Kdiag(self, X):
        v = self.variance * np.exp(np.asarray(X, dtype=float).ravel())
        return np.where(np.isfinite(v), v, 0.0)

    def update_gradients_full(self, dL_dK, X, X2=None):
        pass

    def update_gradients_diag(self, dL_dKdiag, X):
        pass


## Load Silverstone 2011 (FP3 → Q1)

Default demo race in `regressGpPractice`. Cut-off rank **17** is the Q1→Q2
bubble in a 24-car field.


In [ ]:
GRAND_PRIX = "GreatBritain"
PRACTICE = "FP3"
Q_COL = 0  # Y columns: Q1, Q2, Q3
CUTOFF_RANK = 17

if not hasattr(pods.datasets, "f1_2011_practice_qualifying"):
    raise ImportError(
        "pods.datasets.f1_2011_practice_qualifying is missing. "
        "Install pods from ods CIP-0005, e.g. "
        "`python scripts/install_from_yaml.py --profile gpy-tutorials`."
    )

data = pods.datasets.f1_2011_practice_qualifying(
    grand_prix=GRAND_PRIX, practice=PRACTICE
)

p = np.asarray(data["X"], dtype=float).ravel()
Y = np.asarray(data["Y"], dtype=float)
q = Y[:, Q_COL]
driver_num = np.asarray(data["driver_number"], dtype=int).ravel()
team_idx = np.asarray(data["team_index"], dtype=int).ravel()  # 1..12 pit order
abbr = np.asarray(data["abbreviation"])

ok = np.isfinite(p) & np.isfinite(q)
p, q, driver_num, team_idx, abbr = p[ok], q[ok], driver_num[ok], team_idx[ok], abbr[ok]
print(f"{GRAND_PRIX} {PRACTICE}→Q1: {len(p)} drivers with both times")


## Consistency defaults

Percent-style standard deviations are converted to geometric variances with
$\log(1 + (\sigma/100)^2)$, matching MATLAB.


In [ ]:
def pct_var(std_pct):
    std_pct = np.asarray(std_pct, dtype=float)
    return np.log(1.0 + (std_pct * std_pct) / 10000.0)

# Defaults from regressGpPractice.m for Q1/Q2/Q3
m_mean = 0.8
m_std = 2.0
non_lin_std = 0.5
non_lin_width = float(np.std(np.log(p)))
track_std = 5.0
team_std = np.full(12, 0.5)
driver_std = np.ones(len(p))

# Alonso (5) and Massa (6) fixed to observed practice in the 2011 demos
fix_driver = np.isin(driver_num, [5, 6])

driver_var = pct_var(driver_std)  # used for fixed-driver IndexARD scales
Xnoise = np.full(len(p), -np.inf)
Xnoise[~fix_driver] = np.log(np.maximum(driver_var[~fix_driver], 1e-300))

log_p = np.log(p)
# createModelInput columns: [driverNum, teamNum, pLapTimes, logInput, Xnoise]
X = np.column_stack([driver_num, team_idx, p, log_p, Xnoise])
y = (np.log(q) - m_mean * log_p)[:, None]


## Build GP covariance

Hyperparameters are set from the consistency structure. The same compound kernel
is shared by the Gaussian and skew models below.


In [ ]:
team_scales = {t: float(pct_var(team_std[t - 1])) for t in range(1, 13)}
driver_scales = {
    int(d): (float(driver_var[i]) if fix_driver[i] else 0.0)
    for i, d in enumerate(driver_num)
}


def make_kern():
    return (
        GPy.kern.Linear(1, variances=m_std ** 2, active_dims=[3], name="lin")
        + GPy.kern.RBF(
            1,
            variance=float(pct_var(non_lin_std)),
            lengthscale=non_lin_width,
            active_dims=[3],
            name="rbf",
        )
        + GPy.kern.Bias(
            1, variance=float(pct_var(track_std)), active_dims=[3], name="track"
        )
        + IndexARD(team_scales, active_dims=[1], name="team")
        + IndexARD(driver_scales, active_dims=[0], name="driver_fixed")
        + DiagFromInput(1.0, active_dims=[4], name="driver_noise")
    )


kern = make_kern()
print(kern)


## Gaussian vs skew observation noise

Same compound kernel, two likelihoods:

1. **Gaussian** (`GPRegression`) — symmetric residual noise (baseline). Likelihood
   variance is fixed tiny, matching the MATLAB live tool; driver consistency sits
   in `DiagFromInput`.
2. **SkewGaussian** — O’Hagan skew-normal via Laplace approximation. Residual
   *scale* is pinned to a 1% geometric std so optimisation focuses on the skew
   parameter $\alpha$; positive $\alpha$ puts more mass on $y > f$ (slower than
   latent pace).


In [ ]:
model_gauss = GPy.models.GPRegression(X, y, kernel=make_kern(), noise_var=1e-8)
model_gauss.Gaussian_noise.fix()
model_gauss.optimize(messages=False, max_iters=200)

# Pin residual scale; learn alpha (skew direction of non-pace residuals).
skew_var = float(pct_var(1.0))
model_skew = GPy.core.GP(
    X,
    y,
    make_kern(),
    GPy.likelihoods.SkewGaussian(variance=skew_var, alpha=1.5),
    inference_method=GPy.inference.latent_function_inference.Laplace(),
)
model_skew.likelihood.variance.fix()
model_skew.optimize(messages=False, max_iters=200)


def _scalar(x):
    return float(np.asarray(x).reshape(()))


ll_g = _scalar(model_gauss.log_likelihood())
ll_s = _scalar(model_skew.log_likelihood())
print(f"Gaussian  log-likelihood: {ll_g:.3f}")
print(f"Skew      log-likelihood: {ll_s:.3f}")
print(
    f"Skew alpha = {_scalar(model_skew.likelihood.alpha):.3f}  "
    f"variance = {_scalar(model_skew.likelihood.variance):.5f} (fixed)"
)
print()
print("--- Gaussian model ---")
print(model_gauss)
print()
print("--- Skew model ---")
print(model_skew)


## Predicted vs actual Q1 times

Undo the log-link: $\hat{q} = \exp(\hat{y} + m\log p)$. Side-by-side comparison
of Gaussian and skew predictive means (approximate $\pm 2\sigma$ bands).


In [ ]:
def predict_q(model, X_in, log_p_in, m=m_mean):
    # Latent GP mean; observation noise handled by the likelihood separately.
    mu, var = model.predict(X_in, include_likelihood=False)
    mu = np.asarray(mu).ravel()
    var = np.maximum(np.asarray(var).ravel(), 0.0)
    log_q = mu + m * log_p_in
    q_hat = np.exp(log_q)
    q_std = q_hat * np.sqrt(var)  # delta method on latent uncertainty
    return q_hat, q_std


q_g, std_g = predict_q(model_gauss, X, log_p)
q_s, std_s = predict_q(model_skew, X, log_p)

fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharex=True, sharey=True)
lo = min(q.min(), q_g.min(), q_s.min()) - 0.2
hi = max(q.max(), q_g.max(), q_s.max()) + 0.2
for ax, q_hat, q_std, title in [
    (axes[0], q_g, std_g, "Gaussian likelihood"),
    (axes[1], q_s, std_s, "SkewGaussian likelihood"),
]:
    ax.plot([lo, hi], [lo, hi], "k--", lw=1, alpha=0.5)
    ax.errorbar(q_hat, q, xerr=2 * q_std, fmt="o", ms=5, alpha=0.85)
    for a, xh, ya in zip(abbr, q_hat, q):
        ax.annotate(a, (xh, ya), textcoords="offset points", xytext=(4, 2), fontsize=7)
    ax.set_xlabel("Predicted Q1 (s)")
    ax.set_ylabel("Actual Q1 (s)")
    ax.set_title(title)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)
fig.suptitle(f"{GRAND_PRIX} 2011: GP predictions from {PRACTICE}", y=1.02)
plt.tight_layout()
plt.show()

rmse = lambda pred: float(np.sqrt(np.mean((pred - q) ** 2)))
mae = lambda pred: float(np.mean(np.abs(pred - q)))
print(f"{'model':<12} {'RMSE (s)':>10} {'MAE (s)':>10}")
print(f"{'Gaussian':<12} {rmse(q_g):10.4f} {mae(q_g):10.4f}")
print(f"{'Skew':<12} {rmse(q_s):10.4f} {mae(q_s):10.4f}")
print()
print("Gaussian predicted order:", " ".join(abbr[np.argsort(q_g)]))
print("Skew     predicted order:", " ".join(abbr[np.argsort(q_s)]))


## Cut-off distribution (rank 17)

Draw posterior samples of the **latent** function $f$ (`include_likelihood=False`;
`SkewGaussian` does not implement a full observation covariance), map to lap
times, sort each draw, and read off the 17th-fastest time — the Q1 cut-off into
Q2 (`visualizeGrid.m`). Compare Gaussian and skew predictive cut-off histograms.


In [ ]:
rng = np.random.default_rng(1)
n_samp = 2000


def sample_cutoff(model, n_samp=n_samp, rank=CUTOFF_RANK):
    mu, cov = model.predict(X, full_cov=True, include_likelihood=False)
    cov = np.asarray(cov)
    if cov.ndim == 3:
        cov = cov[:, :, 0]
    cov = 0.5 * (cov + cov.T) + 1e-8 * np.eye(len(p))
    y_samp = rng.multivariate_normal(np.asarray(mu).ravel(), cov, size=n_samp).T
    q_samp = np.exp(y_samp + m_mean * log_p[:, None])
    cutoff = np.sort(q_samp, axis=0)[rank - 1, :]
    danger = np.mean(q_samp > cutoff[None, :], axis=1)
    return q_samp, cutoff, danger


q_samp_g, cutoff_g, danger_g = sample_cutoff(model_gauss)
q_samp_s, cutoff_s, danger_s = sample_cutoff(model_skew)

print(f"{'model':<12} {'mean P17':>10} {'std P17':>10}")
print(f"{'Gaussian':<12} {cutoff_g.mean():10.3f} {cutoff_g.std():10.4f}")
print(f"{'Skew':<12} {cutoff_s.mean():10.3f} {cutoff_s.std():10.4f}")
print()

rank_by_mean = np.argsort(q_samp_s.mean(axis=1))
print(f"{'Rk':>3} {'Drv':>4} {'E[Q1] skew':>11} {'Danger17 G':>10} {'Danger17 S':>10}")
for r, i in enumerate(rank_by_mean, start=1):
    print(
        f"{r:3d} {abbr[i]:>4} {q_samp_s.mean(axis=1)[i]:11.3f} "
        f"{100 * danger_g[i]:9.1f}% {100 * danger_s[i]:9.1f}%"
    )

fig, ax = plt.subplots(figsize=(7.5, 3.8))
ax.hist(cutoff_g, bins=40, density=True, color="C0", alpha=0.55, label="Gaussian")
ax.hist(cutoff_s, bins=40, density=True, color="C1", alpha=0.55, label="SkewGaussian")
ax.axvline(cutoff_g.mean(), color="C0", lw=1.5, ls="--")
ax.axvline(cutoff_s.mean(), color="C1", lw=1.5, ls="--")
ax.set_xlabel(f"Sampled P{CUTOFF_RANK} cut-off time (s)")
ax.set_ylabel("Density")
ax.set_title(f"{GRAND_PRIX} 2011 Q1 cut-off from {PRACTICE}: Gaussian vs skew")
ax.legend()
plt.show()

cutoff_mean = float(cutoff_s.mean())
cutoff_std = float(cutoff_s.std())
model = model_skew


## Partial observations (cut-off tool spirit)

`sampleGrids.m` conditions on drivers who have already set a flying lap.
Here we pretend the top 10 Q1 times are known and re-predict the cut-off under
both likelihoods.


In [ ]:
known = np.argsort(q)[:10]
X_k, y_k = X[known], y[known]


def partial_cutoff(make_model, label):
    m = make_model()
    mu_all, cov_all = m.predict(X, full_cov=True, include_likelihood=False)
    cov_all = np.asarray(cov_all)
    if cov_all.ndim == 3:
        cov_all = cov_all[:, :, 0]
    cov_all = 0.5 * (cov_all + cov_all.T) + 1e-8 * np.eye(len(p))
    y_s = rng.multivariate_normal(np.asarray(mu_all).ravel(), cov_all, size=n_samp).T
    q_part = np.exp(y_s + m_mean * log_p[:, None])
    for i in known:
        q_part[i, :] = np.minimum(q_part[i, :], q[i])
    cut = np.sort(q_part, axis=0)[CUTOFF_RANK - 1]
    print(
        f"{label}: cut-off mean {cut.mean():.3f}s  std {cut.std():.4f}s "
        f"(full-data skew cut-off was {cutoff_mean:.3f}s)"
    )
    return cut


def _gauss_partial():
    m = GPy.models.GPRegression(X_k, y_k, kernel=make_kern(), noise_var=1e-8)
    m.Gaussian_noise.fix()
    return m


def _skew_partial():
    m = GPy.core.GP(
        X_k,
        y_k,
        make_kern(),
        GPy.likelihoods.SkewGaussian(variance=skew_var, alpha=1.5),
        inference_method=GPy.inference.latent_function_inference.Laplace(),
    )
    m.likelihood.variance.fix()
    return m


cut_g = partial_cutoff(_gauss_partial, "Gaussian (top-10 observed)")
cut_s = partial_cutoff(_skew_partial, "Skew     (top-10 observed)")


## Notes

- Team indices come from the curated pack (`team_index`, pit-lane order 1–12),
  matching `num2teamNum` in the MATLAB demos.
- Fixed drivers 5 and 6 follow the 2011 demo defaults; change `fix_driver` to
  experiment.
- Positive learned $\alpha$ on the skew model is the quantity of interest: it
  should put more residual mass on the slow side of latent pace.
- A fuller port of the live GUI (`cutoffPredictor`) would stream sector
  updates; this notebook is the static GP + cut-off sampling core.
- Data: `pods` /
  [datasets_mirror `f1_2011_practice_qualifying`](https://github.com/lawrennd/datasets_mirror/tree/main/f1_2011_practice_qualifying)
  ([ods CIP-0005](https://github.com/lawrennd/ods)).
- Likelihoods: [GPy CIP-0014](https://github.com/SheffieldML/GPy) (`SkewGaussian`).
